In [20]:
import polars as pl
import random
import os

DATASET_PATH = "datasets/dataset_merged_with_families.parquet"
CACHE_PATH = "datasets/dataset_merged_with_fixed_families.parquet"
BASE_DIR = "datasets"
FILE_PREFIX = "dataset_homology_split_rev_fixed_"
FULL_PREFIX = os.path.join(BASE_DIR, FILE_PREFIX)
SEED = 42

RENAME_MAP = {
    "original_seq_full": "wt_sequence",
    "mutated_seq_full": "mut_sequence",
    "target": "fitness",
    "mut_type": "mutation",
    "cath_dominant": "cluster_id"
}

SELECTED_COLUMNS = ["wt_sequence", "mut_sequence", "mutation", "fitness", "cath_class", "cath_arch", "cath_topology",
                   "cath_homology", "data_source", "reverse"]

os.makedirs(BASE_DIR, exist_ok=True)

In [21]:
if os.path.exists(CACHE_PATH):
    print(f"Načítám fixnutý dataset z cache: {CACHE_PATH}")
    df_fixed = pl.read_parquet(CACHE_PATH)
else:
    print("Cache nenalezena. Spouštím fixaci rodin (join)...")
    lf = pl.scan_parquet(DATASET_PATH)

    # 1. Mapa: original_seq_full -> cath_dominant (pro reverse=False)
    wt_family_map = (
        lf.filter(pl.col("reverse") == False)
        .select(["original_seq_full", "cath_dominant"])
        .unique(subset=["original_seq_full"])
        .rename({"cath_dominant": "parent_cath_dominant"})
    )

    # 2. Joineme dataset k této mapě
    df_fixed = (
        lf.join(
            wt_family_map,
            left_on="mutated_seq_full",
            right_on="original_seq_full",
            how="left"
        )
        .with_columns([
            pl.when((pl.col("reverse") == True) & (pl.col("parent_cath_dominant").is_not_null()))
            .then(pl.col("parent_cath_dominant"))
            .otherwise(pl.col("cath_dominant"))
            .alias("cath_dominant")
        ])
        .drop("parent_cath_dominant")
        .collect()
    )
    
    print("Aktualizuji CATH úrovně...")
    df_fixed = df_fixed.with_columns([
        pl.col("cath_dominant").str.split(".").alias("_temp_split")
    ])
    df_fixed = df_fixed.with_columns([
        pl.col("_temp_split").list.get(0).alias("cath_class"),
        pl.col("_temp_split").list.get(1).alias("cath_arch"),
        pl.col("_temp_split").list.get(2).alias("cath_topology"),
        pl.col("_temp_split").list.get(3).alias("cath_homology"),
    ]).drop("_temp_split")

    print(f"Ukládám fixnutý dataset do cache: {CACHE_PATH}")
    df_fixed.write_parquet(CACHE_PATH)

print(f"Dataset připraven. Celkem řádků: {len(df_fixed)}")

Načítám fixnutý dataset z cache: datasets/dataset_merged_with_fixed_families.parquet
Dataset připraven. Celkem řádků: 1738861


In [22]:
print("Zahajuji split podle rodin (cath_dominant)...")

# 1. Příprava rodin
family_counts = (
    df_fixed.group_by("cath_dominant")
    .len()
    .sort("cath_dominant")
    .to_dicts()
)

random.seed(SEED)
random.shuffle(family_counts)

total_rows = len(df_fixed)
holdout_goal = int(total_rows * 0.1)
train_fams, holdout_fams = [], []
curr_holdout = 0

for fam in family_counts:
    if curr_holdout < holdout_goal:
        holdout_fams.append(fam["cath_dominant"])
        curr_holdout += fam["len"]
    else:
        train_fams.append(fam["cath_dominant"])

train_df = df_fixed.filter(pl.col("cath_dominant").is_in(train_fams))
holdout_df = df_fixed.filter(pl.col("cath_dominant").is_in(holdout_fams))

# 2. Split Holdout na Val/Test (Test nastaven na 0)
val_df = holdout_df.slice(0, 0)
test_df = holdout_df

print(f"{'TRAIN':<10} | {len(train_df):>8} | {train_df['cath_dominant'].n_unique():>8} rodin | {len(train_df) / total_rows * 100:>5.1f}%")
print(f"{'VAL':<10} | {len(val_df):>8} | {val_df['cath_dominant'].n_unique():>8} rodin | {len(val_df) / total_rows * 100:>5.1f}%")
print(f"{'TEST':<10} | {len(test_df):>8} | {test_df['cath_dominant'].n_unique():>8} rodin | {len(test_df) / total_rows * 100:>5.1f}%")

Zahajuji split podle rodin (cath_dominant)...
TRAIN      |  1560811 |      150 rodin |  89.8%
VAL        |        0 |        0 rodin |   0.0%
TEST       |   178050 |       26 rodin |  10.2%


In [23]:
def save(d, name):
    if len(d) == 0:
        print(f"Set '{name}' je prázdný, přeskakuji zápis.")
        return
    valid_map = {k: v for k, v in RENAME_MAP.items() if k in d.columns}
    d.rename(valid_map).select(SELECTED_COLUMNS).write_csv(f"{FULL_PREFIX}{name}.csv")

print("Ukládám soubory CSV...")
save(train_df, "train")
save(val_df, "validation")
save(test_df, "test")

print(f"Hotovo. Dataset uložen do {BASE_DIR} s prefixem {FILE_PREFIX}")

Ukládám soubory CSV...
Set 'validation' je prázdný, přeskakuji zápis.
Hotovo. Dataset uložen do datasets s prefixem dataset_homology_split_rev_fixed_
